# Imports

In [ ]:
import numpy as np # A useful package for dealing with mathematical processes, we will be using it this week for vectors and matrices
import pandas as pd # A common package for viewing tabular data
import sklearn.linear_model, sklearn.datasets # We want to be able to access the sklearn datasets again, also we are using some model evaluation
from sklearn.preprocessing import StandardScaler, MinMaxScaler # We will be using the imbuilt sclaing functions sklearn provides
import matplotlib.pyplot as plt # We will be using Matplotlib for our graphs
from sklearn.preprocessing import PolynomialFeatures # A preprocessing function allowing us to include polynomial features into our model
from google.colab import files # We will be importing a csv file I have provided for one section.
from sklearn.preprocessing import LabelEncoder, OneHotEncoder # We will be using these to encode categorical features
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import seaborn as sns; sns.set()
from sklearn.metrics import  accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, mean_squared_error, r2_score , confusion_matrix, ConfusionMatrixDisplay # required for evaluating classification models
import joblib
from google.colab import files


# Loading Datasets


In [ ]:
uploaded = files.upload()
front_squat_angle_data = pd.read_csv('front.csv')
side_squat_angle_data = pd.read_csv('side.csv')

Saving side.csv to side (1).csv
Saving front.csv to front (1).csv


# Data Processing


In [ ]:
# load both CSVs into dataframes
front_df = pd.read_csv('front.csv')
side_df = pd.read_csv('side.csv')

# check the shape and first few rows of each dataset
print("Front CSV shape:", front_df.shape)
display(front_df.head())

print("Side CSV shape:", side_df.shape)
display(side_df.head())

# check how many rows per label
print("\nFront label distribution:")
display(front_df['class'].value_counts())

print("\nSide label distribution:")
display(side_df['class'].value_counts())

Front CSV shape: (4060, 13)


,file,class,left_knee_angle,left_hip_angle,left_trunk_angle,right_knee_angle,right_hip_angle,right_trunk_angle,knee_distance,ankle_distance,knee_ankle_ratio,left_knee_foot_offset,right_knee_foot_offset
0,squat_front_1.mp4,good,178.08,172.21,173.15,176.78,172.25,173.74,0.076234,0.083664,0.911195,0.006732,0.000698
1,squat_front_1.mp4,good,177.90,172.08,173.10,176.84,172.25,173.71,0.076564,0.083840,0.913216,0.006412,0.000864
2,squat_front_1.mp4,good,177.70,171.94,173.04,176.85,172.24,173.71,0.076887,0.083934,0.916047,0.006111,0.000935
3,squat_front_1.mp4,good,177.60,171.86,173.01,176.87,172.25,173.70,0.077051,0.084042,0.916810,0.005969,0.001022
4,squat_front_1.mp4,good,177.59,171.84,172.99,176.89,172.24,173.68,0.077100,0.084119,0.916554,0.005947,0.001072


Side CSV shape: (4044, 13)


,file,class,left_knee_angle,left_hip_angle,left_trunk_angle,right_knee_angle,right_hip_angle,right_trunk_angle,knee_distance,ankle_distance,knee_ankle_ratio,left_knee_foot_offset,right_knee_foot_offset
0,squat_side_1.mp4,good,179.33,177.26,177.58,178.55,178.10,178.79,0.034627,0.034307,1.009335,0.011281,0.011601
1,squat_side_1.mp4,good,179.63,176.66,176.84,178.73,178.22,178.83,0.034619,0.035840,0.965957,0.012300,0.011080
2,squat_side_1.mp4,good,179.81,176.25,176.34,178.83,178.36,178.92,0.034727,0.037004,0.938460,0.013047,0.010770
3,squat_side_1.mp4,good,179.91,176.20,176.25,179.41,178.80,179.08,0.033869,0.037637,0.899875,0.013616,0.009848
4,squat_side_1.mp4,good,179.91,176.23,176.27,179.46,178.99,179.25,0.033767,0.037791,0.893519,0.013709,0.009685



Front label distribution:


,count
class,
good,2254
knees_in,1806



Side label distribution:


,count
class,
good,2065
leaning_forward,1979


# Front Model


In [ ]:
# separate features and labels for front view
# drop file and class columns as they are not features
X_front = front_squat_angle_data.drop(columns=['file', 'class'])
y_front = front_squat_angle_data['class']

# encode the labels into numbers (good = 0, knees_in = 1)
le_front = LabelEncoder()
y_front = le_front.fit_transform(y_front)

# split into 80% training and 20% testing
X_front_train, X_front_test, y_front_train, y_front_test = train_test_split(
    X_front, y_front, test_size=0.2, random_state=42
)

# scale the features so all angles are on the same scale
scaler_front = MinMaxScaler()
X_front_train = scaler_front.fit_transform(X_front_train)
X_front_test  = scaler_front.transform(X_front_test)

print("Front view data ready for training!")
print(f"Training samples : {len(X_front_train)}")
print(f"Testing samples  : {len(X_front_test)}")
print(f"Classes          : {le_front.classes_}")

Front view data ready for training!
Training samples : 3248
Testing samples  : 812
Classes          : ['good' 'knees_in']


##